# Blue Team Lab (Network-centric)
## 60-minute investigation using network events (DNS, HTTP, TLS, CONN)

### Scenario
You are a blue team analyst reviewing network telemetry from a short time window.

### Goals
- Identify the most likely compromised host
- Identify suspicious infrastructure (domain/IP)
- Build a timeline of activity
- Recommend containment and one detection improvement
- Document your findings clearly

### Deliverables
1) Answers to the 8 investigation questions
2) A 6–10 sentence incident summary


In [ ]:
import pandas as pd
import numpy as np

# Colab data source (raw GitHub URL)
NETWORK_CSV_URL = "https://raw.githubusercontent.com/dadirad/blue-team-lab/main/files/network_events.csv"

df = pd.read_csv(NETWORK_CSV_URL)

# Normalize timestamps if present
if "ts" in df.columns:
    df["ts"] = pd.to_datetime(df["ts"], utc=True, errors="coerce")
    df = df.sort_values("ts").reset_index(drop=True)

df.head(10)


## Task 0 (2 minutes): Confirm what data you have
Run the next cell to see what event types exist in this dataset.


In [ ]:
df["event_type"].value_counts(dropna=False)


## Task 1 (8 minutes): Identify the most suspicious host
A suspicious host often shows one or more of:
- Rare domains (low prevalence)
- Suspicious HTTP paths (payloads/executables)
- Repeated short connections to the same destination (beaconing)
- A clear sequence: DNS → TLS/HTTP → follow-on connections

Run the next cell and choose a candidate host (src_ip / src_host).


In [ ]:
def nunique_nonnull(series):
    return series.dropna().nunique()

host_summary = (
    df.groupby(["src_ip", "src_host"], dropna=False)
      .agg(
          first_seen=("ts", "min") if "ts" in df.columns else ("dst_ip", "count"),
          last_seen=("ts", "max") if "ts" in df.columns else ("dst_ip", "count"),
          total_events=("event_type", "count"),
          unique_dst_ips=("dst_ip", "nunique"),
          unique_dns_queries=("query", nunique_nonnull) if "query" in df.columns else ("dst_ip", "nunique"),
          unique_tls_sni=("sni", nunique_nonnull) if "sni" in df.columns else ("dst_ip", "nunique"),
          unique_http_hosts=("http_host", nunique_nonnull) if "http_host" in df.columns else ("dst_ip", "nunique"),
          http_events=("event_type", lambda s: (s=="HTTP").sum()),
          dns_events=("event_type", lambda s: (s=="DNS").sum()),
          tls_events=("event_type", lambda s: (s=="TLS").sum()),
          conn_events=("event_type", lambda s: (s=="CONN").sum()),
      )
      .reset_index()
      .sort_values(["total_events", "unique_dst_ips"], ascending=False)
)

host_summary


## Task 2 (10 minutes): Find suspicious domains and suspicious HTTP paths
Look for:
- DNS queries that appear only once (rare)
- HTTP URIs that look like payload delivery (.exe, /payload, .ps1, .dll, .bin)
- Unusual hosts/domains tied to those URIs


In [ ]:
if "query" in df.columns:
    dns = df[df["event_type"]=="DNS"].copy()
    dns_counts = dns["query"].dropna().value_counts()
    rare_dns = dns_counts[dns_counts == 1]
    display(rare_dns.head(25))
else:
    print("No 'query' column found. Skipping rare DNS analysis.")


In [ ]:
if "uri" in df.columns and "http_host" in df.columns:
    http = df[df["event_type"]=="HTTP"].copy()
    pattern = r"(payload|\.exe|\.ps1|\.dll|\.bin|\.dat|/download|/update|/install)"
    susp_http = http[http["uri"].fillna("").str.contains(pattern, case=False, regex=True)]
    cols = [c for c in ["ts","src_ip","src_host","dst_ip","dst_port","http_host","uri","method","status","bytes_in","notes"] if c in df.columns]
    display(susp_http[cols].sort_values("ts") if "ts" in df.columns else susp_http[cols])
else:
    print("No HTTP columns (uri/http_host) found. Skipping suspicious HTTP analysis.")


## Task 3 (10 minutes): Look for beaconing behavior
Beaconing often looks like repeated connections from one host to one destination at a regular interval.
Run the next cell and identify any candidate beacon flows.


In [ ]:
required = {"src_ip","dst_ip","dst_port","event_type"}
if required.issubset(df.columns) and "ts" in df.columns:
    conn = df[df["event_type"]=="CONN"].copy()
    conn = conn.sort_values(["src_ip","dst_ip","dst_port","ts"])
    conn["delta_sec"] = conn.groupby(["src_ip","dst_ip","dst_port"])["ts"].diff().dt.total_seconds()

    group_cols = [c for c in ["src_ip","src_host","dst_ip","dst_port"] if c in conn.columns]
    beacon_stats = (
        conn.groupby(group_cols, dropna=False)
            .agg(
                count=("ts","count"),
                mean_delta=("delta_sec","mean"),
                std_delta=("delta_sec","std"),
                first=("ts","min"),
                last=("ts","max")
            )
            .reset_index()
    )

    beacon_candidates = beacon_stats[(beacon_stats["count"]>=4) & (beacon_stats["mean_delta"].notna())].copy()
    beacon_candidates = beacon_candidates.sort_values(["count","std_delta"], ascending=[False, True])
    display(beacon_candidates.head(25))
else:
    print("Beaconing analysis requires src_ip, dst_ip, dst_port, event_type, and ts columns. Skipping.")


## Task 4 (15 minutes): Build the timeline for your chosen host
1) Set HOST_IP to your suspected host
2) Run the cell to see all events for that host
3) Identify: first suspicious DNS, any TLS SNI, any HTTP download-like requests, follow-on infra, and repeated connections.


In [ ]:
HOST_IP = ""  # e.g., "10.0.5.23"
if not HOST_IP:
    print("Set HOST_IP to a value like '10.0.5.23' and re-run this cell.")
else:
    host_events = df[df["src_ip"]==HOST_IP].copy()
    cols = [c for c in ["ts","event_type","src_ip","src_host","dst_ip","dst_port","query","sni","http_host","uri","status","bytes_in","notes"] if c in df.columns]
    display(host_events[cols].sort_values("ts") if "ts" in df.columns else host_events[cols])


In [ ]:
SUSP_DOMAIN = ""  # optional: set to a suspicious domain and re-run
if not SUSP_DOMAIN:
    print("Optional: set SUSP_DOMAIN to pivot across DNS/TLS/HTTP and re-run.")
else:
    mask = False
    if "query" in df.columns:
        mask = mask | (df["query"]==SUSP_DOMAIN)
    if "sni" in df.columns:
        mask = mask | (df["sni"]==SUSP_DOMAIN)
    if "http_host" in df.columns:
        mask = mask | (df["http_host"]==SUSP_DOMAIN)

    domain_events = df[mask].copy()
    cols = [c for c in ["ts","event_type","src_ip","src_host","dst_ip","dst_port","query","sni","http_host","uri","status","bytes_in","notes"] if c in df.columns]
    display(domain_events[cols].sort_values("ts") if "ts" in df.columns else domain_events[cols])


## Submit: 8 Investigation Questions
1) Suspected compromised host (IP + hostname):
2) Most suspicious external destination (domain or IP):
3) First suspicious timestamp (with evidence):
4) Primary lead (what triggered your investigation?):
5) Evidence that supports malicious activity (2–3 bullets):
6) Best-guess attack technique (plain language is fine):
7) Immediate containment actions (1–2 actions):
8) One detection improvement idea:

## Incident Summary (6–10 sentences)
**Title:** Suspected malicious activity on `<host>`

- At `<time>`, host `<internal IP>` initiated `<protocol>` activity to `<domain/IP>`.
- Evidence includes `<DNS query / TLS SNI / HTTP URI>` and supporting connection patterns.
- The sequence suggests `<download / beaconing>` within the capture window.
- Follow-on traffic to `<second infrastructure>` indicates possible command and control.
- Confidence: `<high/medium/low>` because `<reason>`.
- Recommended containment: `<actions>`.
- Detection improvement: `<specific rule/tuning idea>`.
